### Import packages
#### Create connection to Database

In [1]:
from sqlalchemy import create_engine
import pandas as pd

# Connect to the database
db_connection_string = 'sqlite:///chinook.db'
db_engine = create_engine(url=db_connection_string)
db_conn = db_engine.connect()

#### Read table from database

In [ ]:
# HINTS
# Load data into DataFrame
# user Pandas to read data from a table into a DataFrame

In [ ]:
# Approach 1: Use Pandas.read_sql_table to read all columns from 'customers' table
table_name = 'customers'
df = pd.read_sql_table(table_name, con=db_conn)
df.tail(5)

In [ ]:
# # Approach 2: Use Pandas.read_sql_query to read these columns
table_name = 'customers'
columns = 'CustomerId, FirstName, LastName, Phone, Email, SupportRepId'
df = pd.read_sql_query(sql=f'select {columns} from {table_name}',con=db_conn)
df.tail(5)

#### Config-Driven Ingestion

In [ ]:
# # HINTS
# # Read configs stored in the 'config.yml' file
# Use loop function to read tables within config.source.table
# Export output into CSV
# Name Convention: '<date>__<table_name>.csv'
# Path: Destination/config_driven/
# note: use os.makedirs() if path is not exists


# # Read yaml file
# # Package: yaml (pip install pyyaml)
# # Function: load / safe_load
# # Print it after loading

import os
import io
import yaml
import datetime

config_file = 'config.yml'
f = open(config_file, 'r')
# Use safe_load for security,  only allows the creation of basic Python objects (like integers, lists, and dictionaries)
# prevents the execution of arbitrary code from untrusted sources
# it converts YAML to a Python dictionary: 
config = yaml.safe_load(f) 
print(config)

export_date=datetime.datetime.now()
export_date=export_date.strftime("%Y%m%d")

def extract_table(table_name, con, folder_path):
    os.makedirs(folder_path, exist_ok=True)
    print(f'Extracting {table_name} ...')
    df = pd.read_sql_table(table_name=table_name, con=con)
    df.to_csv(f'{folder_path}/{export_date}_{table_name}.csv',index=False)
    print('Completed!\n')

def get_connection(db_type, host):
    if db_type == 'sqlite':
        db_connection_string = f'sqlite:///{host}.db'
        db_engine = create_engine(url=db_connection_string)
        return db_engine.connect()
    elif db_type == 'Oracle':
        db_connection_string = 'Oracle://{host}:1234'
        return db_engine.connect()
db_conn = get_connection(**config.get('source').get('database'))

# Use loop function to read tables within config.source.table
for table_name in config.get('source').get('table'):
    extract_table(table_name=table_name, con=db_conn, folder_path='destination/config_driven')

In [ ]:

# Export output into CSV
# Name Convention: '<date>__<table_name>.csv'
# Path: Destination/config_driven/
# note: use os.makedirs() if path is not exists

def extract_table(table_name, con, folder_path):
    os.makedirs(folder_path)
    print(f'Extracting {table_name} ...')
    df = pd.read_sql_table(table_name=table_name, con=db_conn)
    df.to_csv(f'{folder_path}/{table_name}.csv')
    print('Completed!\n')

def get_connection(db_type, host):
    if db_type == 'sqlite':
        db_connection_string = f'sqlite:///{host}.db'
        db_engine = create_engine(url=db_connection_string)
        return db_engine.connect()
    elif db_type == 'Oracle':
        db_connection_string = 'Oracle://{host}:1234'
        return db_engine.connect()
db_conn = get_connection(**config.get('source').get('database'))

### Metadata-Driven Ingestion

In [ ]:
# # HINTS
# # Read metadata from the database inlcuding tables / columns
# loop for each table from the DataFrame
# read and extract table
# save to path: destination/metadata_driven/
# note: use os.makedirs() if path is not exists


#Metadata-driven Ingestion - Approach 2: Use Pandas.read_sql_query to read these columns

import os
import datetime

sqlite_metadata_table = 'sqlite_master'
sqlite_metadata_condition = "type = 'table'"
metadata_sql=f'select tbl_name from {sqlite_metadata_table} where {sqlite_metadata_condition}'

export_date=datetime.datetime.now()
export_date=export_date.strftime("%Y%m%d")
table_df = pd.read_sql_query(metadata_sql,con=db_conn)
folder_path=f'destination/metadata_driven/{export_date}'
os.makedirs(folder_path,exist_ok=True)
for i in range(len(table_df)):
    table_name = str(table_df.loc[i].tbl_name)
    # Get the current date and time and format 
    current_datetime = datetime.datetime.now()
    print(current_datetime)
    print(f'Exporting {table_name}...')
    # df = pd.read_sql_table(table_name, con=db_conn)
    df = pd.read_sql_query(str("select * from " + table_name), con=db_conn)
    export_file_name=current_datetime.strftime("%Y%m%d") + '_' + table_name + '.csv'  
    df.to_csv(f'{folder_path}/{export_file_name}', index=False)
    # clear dataframe of extracted table
    df.drop(df.columns, axis=1, inplace=True)
    print('Completed')

In [ ]:
# Metadata-driven Ingestion -Approach 1: Use Pandas.read_sql_table to read all columns
import os 

metadata_sql = """select name from sqlite_master where 1=1 and type = 'table'"""
table_df = pd.read_sql_query(metadata_sql, con=db_conn)
names = [tb for tb in list(table_df['name']) if tb not in ('sqlite_stat1', 'sqlite_sequence')]

# loop for each table from the DataFrame
# read and extract table
# save to path: chinook/metadata_driven/
# note: use os.makedirs() if path is not exists
def extract_table(table_name, con, folder_path):
    os.makedirs(folder_path, exist_ok=True)
    print(f'Extracting {table_name} ...')
    df = pd.read_sql_table(table_name=table_name, con=db_conn)
    df.to_csv(f'{folder_path}/{table_name}.csv', index=False)
    print('Completed!\n')
for name in names:
    extract_table(table_name=name, con=db_conn, folder_path='destination/metadata')

In [ ]:
# Metadata-driven Ingestion: use inspect from sqlalchemy to check table names in database
from sqlalchemy import create_engine, inspect
import pandas as pd
import os
# Connect to the database
db_connection_string = 'sqlite:///chinook.db'
db_engine = create_engine(url=db_connection_string)
db_conn = db_engine.connect()
db_inspector = inspect(db_engine)

tables_df = db_inspector.get_table_names()
print("Table found: ", tables_df)


output_folder = 'destination/metadata/exports'
os.makedirs(output_folder, exist_ok=True)
for table_name in tables_df:
    file_path = os.path.join(output_folder)
    df.to_csv(f"{output_folder}/{table_name}.csv", index = False)
    print(f"Exported: {table_name}")